# Native SmolVLM-256M no-pruner finetune
Two Kaggle GPUs, FP32, per-rank batch 4, accumulation 8 (effective batch 64). The frozen native vision encoder produces 1024 x 768 raw features per camera; the pretrained connector produces 64 x 576 tokens. All 384 visual tokens enter the causal language model. Connector, language model and head train from pretrained weights; there is no alignment stage.

Run cells in order. Full raw feature cache is about 35.8 GiB for 4072 distinct camera groups, plus prepared data, checkpoints and a required 6 GiB free reserve. Cache is created/downloaded in /kaggle/tmp/cache_smolvlm, then saved as public Kaggle Dataset cache_smolvlm. Later sessions read that cache directly from Input. Prepared data stays separate (local or its existing HF repo); it is never uploaded with the cache. Required space is checked on the actual cache and output filesystems. GPU fit, throughput and quality remain unmeasured. A valid cache plus matching prepared JSON does not require raw photos.


In [ ]:
import os, json, sys
os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
GRADIENT_CHECKPOINTING = True
if type(GRADIENT_CHECKPOINTING) is not bool:
    raise TypeError('GRADIENT_CHECKPOINTING must be a bool')
GITHUB_REPO_URL = os.environ.get('GITHUB_REPO_URL', 'https://github.com/huyvanzzz/Group_HND_car.git')
GITHUB_BRANCH = os.environ.get('GITHUB_BRANCH', 'huy-smolvlm-256m-no-pruner')
PREPARED_DATA_HF_REPO_ID = ''  # Dataset repo: manifest.json, train.jsonl, val.jsonl, test.jsonl at root.
CACHE_HF_REPO_ID = ''  # Repo HF cache: features.float16.memmap, index.json, manifest.json.
CACHE_KAGGLE_DATASET = ''  # Lan sau: 'your_kaggle_username/cache-smolvlm'; lan dau de trong.
CACHE_TMP_DIR = '/kaggle/tmp/cache_smolvlm'
FINETUNE_CHECKPOINT_HF_REPO_ID = ''  # Optional model repo: checkpoint plus <filename>.rng/**.
FINETUNE_CHECKPOINT_FILENAME = 'finetune_latest.pt'
if PREPARED_DATA_HF_REPO_ID and PREPARED_DATA_HF_REPO_ID == CACHE_HF_REPO_ID:
    raise ValueError('Prepared data and raw vision cache need separate dataset repositories')
os.environ['HF_HUB_DISABLE_XET'] = '1'
os.environ['HF_HUB_DOWNLOAD_TIMEOUT'] = '120'
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret('HF_TOKEN')
    if token:
        os.environ['HF_TOKEN'] = token
except Exception as exc:
    print('HF_TOKEN was not loaded:', type(exc).__name__)
print({'branch': GITHUB_BRANCH, 'gradient_checkpointing': GRADIENT_CHECKPOINTING,
       'hf_token_present': bool(os.environ.get('HF_TOKEN'))})

# Them Secrets KAGGLE_API_TOKEN (hoac KAGGLE_USERNAME + KAGGLE_KEY) de publish.
# Khong can token publish neu chi doc cache public tu Kaggle Input.
if not CACHE_KAGGLE_DATASET:
    for secret_name in ('KAGGLE_API_TOKEN', 'KAGGLE_USERNAME', 'KAGGLE_KEY'):
        if not os.environ.get(secret_name):
            try:
                value = UserSecretsClient().get_secret(secret_name)
                if value:
                    os.environ[secret_name] = value
            except Exception:
                pass  # authenticate_publisher se bao loi thieu credentials truoc tai cache.


In [ ]:
import subprocess
if sys.version_info < (3, 11):
    raise RuntimeError('Kaggle storage dependencies require Python >=3.11')
from pathlib import Path
REPO = Path('/kaggle/working/Efficient_VLM_For_Autonomous_Driving')
if not (REPO / '.git').is_dir():
    subprocess.run(['git', 'clone', '--branch', GITHUB_BRANCH, GITHUB_REPO_URL, str(REPO)], check=True)
subprocess.run(['git', 'fetch', 'origin', GITHUB_BRANCH], cwd=REPO, check=True)
subprocess.run(['git', 'checkout', GITHUB_BRANCH], cwd=REPO, check=True)
subprocess.run(['git', 'pull', '--ff-only', 'origin', GITHUB_BRANCH], cwd=REPO, check=True)
os.chdir(REPO)
subprocess.run([sys.executable, '-m', 'pip', 'install', '--only-binary=tokenizers', '-e', '.',
                'transformers==4.46.3', 'tokenizers==0.20.4', 'accelerate==1.15.0', 'kaggle==2.2.5', 'kagglehub==1.0.3', 'pycocoevalcap'], check=True)
sys.path.insert(0, str(REPO / 'src'))
os.environ['PYTHONPATH'] = str(REPO / 'src')
# Kaggle includes unrelated apps (Gradio/LiteLLM/Diffusers) with different pins.
# Report their conflicts, but fail on ANY broken dependency in our workflow tree.
from efficient_vlm_ad.smolvlm.environment import validate_dependencies
workflow_requirements = [
    'efficient-vlm-ad', 'transformers==4.46.3', 'tokenizers==0.20.4',
    'accelerate==1.15.0', 'kaggle==2.2.5', 'kagglehub==1.0.3', 'pycocoevalcap',
]
pip_report = subprocess.run([sys.executable, '-m', 'pip', 'check'],
                            capture_output=True, text=True)
print(pip_report.stdout)
if pip_report.stderr:
    print(pip_report.stderr)
if pip_report.returncode not in (0, 1):
    raise RuntimeError('pip check failed to execute; inspect the output above')
workflow_versions = validate_dependencies(workflow_requirements)
print('[ENV] SmolVLM dependency tree OK:', workflow_versions)
if pip_report.returncode:
    print('[ENV] Global pip check reports conflicts outside the validated workflow tree.')
from transformers import AutoProcessor, Idefics3Config, Idefics3ForConditionalGeneration
import torch, transformers, tokenizers, accelerate
assert transformers.__version__ == '4.46.3'
assert tokenizers.__version__ == '0.20.4'
assert accelerate.__version__ == '1.15.0'
assert torch.cuda.is_available() and torch.cuda.device_count() >= 2, 'Enable two Kaggle GPUs'
print([torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
subprocess.run(['nvidia-smi'], check=True)


In [ ]:
import yaml
from dataclasses import replace
from huggingface_hub import snapshot_download
from efficient_vlm_ad.smolvlm.config import load_config
from efficient_vlm_ad.smolvlm.storage import ensure_prepared, resolve_cache
from efficient_vlm_ad.smolvlm.publishing import authenticate_publisher, preflight_dataset, publish_cache
from efficient_vlm_ad.debugging import DebugPrinter
TRACKED_CONFIG = Path('configs/smolvlm_256m_kaggle_2gpu_no_pruner.yaml')
RUNTIME_CONFIG = Path('/kaggle/working/smolvlm_no_pruner_runtime.yaml')
raw = yaml.safe_load(TRACKED_CONFIG.read_text(encoding='utf-8'))
if type(GRADIENT_CHECKPOINTING) is not bool:
    raise TypeError('GRADIENT_CHECKPOINTING must be a bool')
raw['training']['gradient_checkpointing'] = GRADIENT_CHECKPOINTING
cfg = load_config(TRACKED_CONFIG)
cfg = replace(cfg, training=replace(cfg.training, gradient_checkpointing=GRADIENT_CHECKPOINTING))
assert cfg.runtime.precision == 'fp32'
PROFILE = Path(cfg.project.output_dir).resolve()
PROFILE.mkdir(parents=True, exist_ok=True)
publish_required = not bool(CACHE_KAGGLE_DATASET)
storage_result = None
cache_location = None
publisher_api = publisher_owner = None
if publish_required:
    publisher_api, publisher_owner = authenticate_publisher()
    preflight_dataset(publisher_api, f'{publisher_owner}/cache-smolvlm')

def assert_storage_ready():
    if storage_result is None or storage_result.get('status') != 'ready':
        raise RuntimeError('Chay cell cache/publish thanh cong truoc diagnose hoac training.')
    if (cache_location is None
            or storage_result.get('cache_dir') != str(cache_location.directory)
            or storage_result.get('fingerprint') != cache_location.manifest['fingerprint']):
        raise RuntimeError('Cache da thay doi; chay lai cell cache va publish.')
    current = yaml.safe_load(RUNTIME_CONFIG.read_text(encoding='utf-8'))
    if current.get('cache', {}).get('dir') != str(cache_location.directory):
        raise RuntimeError('Runtime cache path da thay doi; chay lai cell cache va publish.')

def run_backend(command, *args):
    assert_storage_ready()
    subprocess.run([sys.executable, '-m', 'efficient_vlm_ad.smolvlm', command,
                    '--config', str(RUNTIME_CONFIG), *args], check=True)

print({'config': str(RUNTIME_CONFIG), 'output': str(PROFILE), 'effective_batch': 4 * 8 * 2})


In [ ]:
storage_result = None
cache_location = None
# Prepared data quan ly rieng: giu ban local hop le, hoac tai repo HF da co.
# Khong tao lai prepared data va khong upload prepared data len Kaggle Dataset.
ensure_prepared(cfg, PREPARED_DATA_HF_REPO_ID)
cache_location = resolve_cache(cfg, cache_tmp_dir=CACHE_TMP_DIR,
                               kaggle_dataset=CACHE_KAGGLE_DATASET, hf_repo_id=CACHE_HF_REPO_ID)
raw['cache']['dir'] = str(cache_location.directory)
RUNTIME_CONFIG.write_text(yaml.safe_dump(raw, sort_keys=False), encoding='utf-8')
cfg = load_config(RUNTIME_CONFIG)
# Cache tmp chi ton tai trong phien; Input duoc doc truc tiep, khong copy sang tmp.


In [ ]:
# Dataset public chi co cache + README, khong co prepared_data.
# Neu upload loi: giu nguyen cache va chay lai RIENG cell nay, khong extract lai.
# Lan sau dien handle duoc in ben duoi vao CACHE_KAGGLE_DATASET de doc Input.
storage_result = None
if publish_required:
    storage_result = publish_cache(cache_location.directory, api=publisher_api,
                                   owner=publisher_owner, prepared_repo_id=PREPARED_DATA_HF_REPO_ID)
else:
    storage_result = {'status': 'ready', 'handle': CACHE_KAGGLE_DATASET, 'reused': True}
storage_result.update(cache_dir=str(cache_location.directory), fingerprint=cache_location.manifest['fingerprint'])
assert_storage_ready()
storage_log = DebugPrinter(enabled=True, jsonl_path=PROFILE / 'debug/debug_events.jsonl')
storage_log.log('STORAGE', {**storage_result, 'source': cache_location.source,
                'revision': cache_location.revision, 'cache_dir': str(cache_location.directory),
                'fingerprint': cache_location.manifest['fingerprint']})
print(storage_result)


In [ ]:
run_backend('diagnose-train', '--debug', '--debug-samples', '1', '--debug-numerics')


In [ ]:
# Single process: compare real full-window and tail gradients before training.
run_backend('verify-accumulation', '--debug', '--debug-numerics')


In [ ]:
assert_storage_ready()
checkpoint_dir = PROFILE / 'checkpoints'
checkpoint_dir.mkdir(parents=True, exist_ok=True)
LOCAL_LATEST = checkpoint_dir / 'finetune_latest.pt'
configured_local = checkpoint_dir / FINETUNE_CHECKPOINT_FILENAME
resume_path = None
if LOCAL_LATEST.is_file():
    resume_path = LOCAL_LATEST
elif configured_local.is_file():
    resume_path = configured_local
elif FINETUNE_CHECKPOINT_HF_REPO_ID:
    snapshot_download(repo_id=FINETUNE_CHECKPOINT_HF_REPO_ID, repo_type='model', local_dir=checkpoint_dir,
                      allow_patterns=[FINETUNE_CHECKPOINT_FILENAME, f'{FINETUNE_CHECKPOINT_FILENAME}.rng/**'])
    resume_path = configured_local
resume_args = []
if resume_path is not None:
    for path in (resume_path, Path(str(resume_path) + '.rng/manifest.json'),
                 Path(str(resume_path) + '.rng/rank_00000.pt'), Path(str(resume_path) + '.rng/rank_00001.pt')):
        if not path.is_file():
            raise FileNotFoundError(path)
    manifest = json.loads(Path(str(resume_path) + '.rng/manifest.json').read_text(encoding='utf-8'))
    if manifest['world_size'] != 2:
        raise ValueError('Resume checkpoint must contain two ranks of RNG state')
    # Standalone validator checks stored world size, metadata and sidecars before launch.
    run_backend('verify-resume-checkpoint', '--checkpoint', str(resume_path), '--stage', 'finetune')
    resume_args = ['--resume', str(resume_path)]
else:
    print('Starting from pretrained native vision encoder, connector, causal LM and head.')
subprocess.run([sys.executable, '-m', 'accelerate.commands.launch', '--multi_gpu', '--num_processes', '2',
                '--num_machines', '1', '--mixed_precision', 'no', '--dynamo_backend', 'no',
                '-m', 'efficient_vlm_ad.smolvlm', 'train', '--config', str(RUNTIME_CONFIG),
                '--stage', 'finetune', *resume_args, '--debug', '--debug-samples', '1', '--debug-numerics'], check=True)


In [ ]:
checkpoint = PROFILE / 'checkpoints/finetune_best.pt'
if not checkpoint.is_file():
    checkpoint = PROFILE / 'checkpoints/finetune_latest.pt'
run_backend('evaluate', '--checkpoint', str(checkpoint), '--batch-size', '4')
run_backend('benchmark', '--checkpoint', str(checkpoint), '--max-samples', '200')
for filename in ('metrics.json', 'benchmark.json'):
    print(filename, (PROFILE / filename).read_text(encoding='utf-8'))
print(''.join((PROFILE / 'predictions.jsonl').read_text(encoding='utf-8').splitlines(keepends=True)[:8]))


In [ ]:
import tarfile
archive = Path('/kaggle/working/smolvlm_256m_no_pruner_finetune_artifacts.tar.gz')
# Explicit allowlist excludes the large cache and raw photos.
with tarfile.open(archive, 'w:gz') as bundle:
    bundle.add(RUNTIME_CONFIG, arcname=RUNTIME_CONFIG.name)
    for name in ('checkpoints', 'prepared_data', 'metrics.json', 'benchmark.json', 'predictions.jsonl', 'debug'):
        path = PROFILE / name
        if path.exists():
            bundle.add(path, arcname=f'{PROFILE.name}/{name}')
print(archive, archive.stat().st_size)
